# Evaluation Example

Notebook para evaluar un modelo entrenado sobre el `TrafficDataset`. Calcula métricas por horizonte temporal, incluyendo MSE, MAE, exactitud por nivel (predicción redondeada) y distancia absoluta media.


In [ ]:
cd ..

In [ ]:
from pathlib import Path
from datetime import datetime
import time
import pickle

import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch.utils.data import DataLoader
from torch_geometric_temporal.nn.recurrent import GConvGRU, TGCN, GConvLSTM, A3TGCN2
from tqdm.auto import tqdm
import pandas as pd

from src.sparse_dataloader import TrafficDataset
from src.training_utils import pad_collate
from src.prediction_reconstruction import reconstruct_predictions
import numpy as np

In [ ]:
processed_data_dir = "data/datos_procesados"
sparse_data_dir = f"{processed_data_dir}/sparse_datasets"

checkpoint_path = Path("data/artifacts/traffic_gnn_v4_2025_11_06.pt")
model_type = "GConvGRU"

in_timesteps = 150
out_timesteps = 50
batch_size = 16
use_cpu = False
max_batches_eval = 3  # For testing purposes

device = torch.device("cpu" if use_cpu or not torch.cuda.is_available() else "cuda")
print(f"Evaluando en dispositivo: {device}")


In [ ]:
# dynamic_data_paths = [
#     f"{sparse_data_dir}/coi_points_dataset.h5",
#     f"{sparse_data_dir}/coi_tracts_dataset.h5",
#     f"{sparse_data_dir}/irregularities_dataset.h5",
#     f"{sparse_data_dir}/events_dataset.h5",
#     f"{sparse_data_dir}/incidents_dataset.h5",
#     f"{sparse_data_dir}/jams_variables_dataset.h5",
# ]

# metadata_paths = [
#     f"{sparse_data_dir}/coi_points_dataset_metadata.pkl",
#     f"{sparse_data_dir}/coi_tracts_dataset_metadata.pkl",
#     f"{sparse_data_dir}/irregularities_dataset_metadata.pkl",
#     f"{sparse_data_dir}/events_dataset_metadata.pkl",
#     f"{sparse_data_dir}/incidents_dataset_metadata.pkl",
#     f"{sparse_data_dir}/jams_variables_dataset_metadata.pkl",
# ]

# future_features = [
#     "coi_puntos",
#     "coi_tramo",
#     "AGLOMERACION",
#     "EMERGENCIA",
#     "MANIFESTACION",
# ]

# traffic_dataset = TrafficDataset(
#     graph_path=f"{processed_data_dir}/conectivity_graph/road_graph.pickle",
#     static_data_path=f"{sparse_data_dir}/static_dataset.h5",
#     dynamic_data_paths=dynamic_data_paths,
#     metadata_paths=metadata_paths,
#     target_data_path=f"{sparse_data_dir}/jams_dataset.h5",
#     in_timesteps=in_timesteps,
#     out_timesteps=out_timesteps,
#     future_features=future_features,
#     node_batch_size=500,
#     weather_data_dir=sparse_data_dir,
#     include_weather=True,
#     neighbor_features_path=f"{processed_data_dir}/neighbor_features.parquet",
# )

with open("data/artifacts/traffic_dataset_v4.pkl", "rb") as f:
    traffic_dataset = pickle.load(f)

In [ ]:
traffic_dataloader = DataLoader(
    traffic_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=pad_collate,
)

print(f"Samples: {len(traffic_dataset)} | Batches: {len(traffic_dataloader)}")
print(f"Static dim: {traffic_dataset.num_static_features}")
print(f"Dynamic past dim: {traffic_dataset.num_dynamic_features}")
print(f"Weather dim: {traffic_dataset.num_weather_features}")
print(f"Future-known dim: {traffic_dataset.num_dynamic_future_features}")

In [ ]:
class StaticEncoder(nn.Module):
    def __init__(self, static_dim, out_dim=32):
        super().__init__()
        self.fc = nn.Linear(static_dim, out_dim)

    def forward(self, x):
        return F.relu(self.fc(x))


def make_batched_edge_index(
    edge_index: torch.Tensor, n_nodes: int, batch_size: int, device
):
    edges = edge_index.size(1)
    repeated = edge_index.repeat(1, batch_size).to(device)
    offsets = (torch.arange(batch_size, device=device) * n_nodes).repeat_interleave(
        edges
    )
    repeated[0] += offsets
    repeated[1] += offsets
    return repeated


class GNN_Temporal_Model(nn.Module):
    def __init__(
        self,
        edge_index,
        static_dim,
        dynamic_dim,
        future_dim,
        input_seq_len,
        forecast_horizon,
        model_type="GConvGRU",
    ):
        super().__init__()
        self.edge_index = None
        self.static_encoder = StaticEncoder(static_dim)

        if model_type == "GConvGRU":
            self.recurrent = GConvGRU(32 + dynamic_dim, 32, K=2)
            self.future_recurrent = GConvGRU(32 + future_dim, 32, K=2)
        elif model_type == "TGCN":
            self.recurrent = TGCN(32 + dynamic_dim, 32)
            self.future_recurrent = TGCN(32 + future_dim, 32)
        elif model_type == "GConvLSTM":
            self.recurrent = GConvLSTM(32 + dynamic_dim, 32, K=2)
            self.future_recurrent = GConvLSTM(32 + future_dim, 32, K=2)
        # elif model_type == "A3TGCN":
        #     self.recurrent = A3TGCN2(
        #         in_channels=32 + dynamic_dim,
        #         out_channels=32,
        #         periods=input_seq_len,
        #         kernel=2,
        #     )
        #     self.future_recurrent = A3TGCN2(
        #         in_channels=32 + future_dim,
        #         out_channels=32,
        #         periods=forecast_horizon,
        #         kernel=2,
        #     )
        else:
            raise ValueError(f"Unknown model type: {model_type}")

        self.linear = nn.Linear(32, 1)
        self.model_type = model_type

    def forward(
        self, edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs
    ):
        batch_size, tin, n_nodes, dyn_dim = dynamic_seq_inputs.shape
        horizon = future_known_inputs.shape[1]
        device = static_inputs.device

        if (
            self.edge_index is None
            or self.edge_index.shape[1] != edge_index.shape[1] * batch_size
        ):
            self.edge_index = make_batched_edge_index(
                edge_index, n_nodes, batch_size, device
            )

        static_embed = self.static_encoder(static_inputs)
        static_embed = static_embed.unsqueeze(0).expand(batch_size, n_nodes, 32)

        hidden = torch.zeros(batch_size, n_nodes, 32, device=device)
        hidden_flat = hidden.reshape(batch_size * n_nodes, 32)

        if self.model_type != "A3TGCN":
            for step in range(tin):
                x_step = dynamic_seq_inputs[:, step]
                combined = torch.cat([static_embed, x_step], dim=-1).reshape(
                    batch_size * n_nodes, 32 + dyn_dim
                )
                hidden_flat = self.recurrent(combined, self.edge_index, H=hidden_flat)
        else:
            seq = []
            for step in range(tin):
                x_step = dynamic_seq_inputs[:, step]
                combined = torch.cat([static_embed, x_step], dim=-1)
                seq.append(combined.reshape(batch_size * n_nodes, -1).unsqueeze(1))
            full_seq = torch.cat(seq, dim=1)
            hidden_flat = self.recurrent(full_seq, self.edge_index)

        outputs = []
        for step in range(horizon):
            f_step = future_known_inputs[:, step]
            combined_future = torch.cat(
                [hidden_flat.view(batch_size, n_nodes, 32), f_step], dim=-1
            )
            combined_future = combined_future.reshape(batch_size * n_nodes, -1)
            hidden_flat = self.future_recurrent(
                combined_future, self.edge_index, H=hidden_flat
            )
            outputs.append(self.linear(hidden_flat).view(batch_size, n_nodes, 1))

        return torch.cat(outputs, dim=-1)

In [ ]:
weather_dim = (
    traffic_dataset.num_weather_features if traffic_dataset.include_weather else 0
)
model = GNN_Temporal_Model(
    edge_index=None,
    static_dim=traffic_dataset.num_static_features,
    dynamic_dim=traffic_dataset.num_dynamic_features + weather_dim,
    future_dim=traffic_dataset.num_dynamic_future_features,
    input_seq_len=in_timesteps,
    forecast_horizon=out_timesteps,
    model_type=model_type,
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()
print(f"Checkpoint cargado desde {checkpoint_path}")

In [ ]:
def evaluate_batch(preds, targets):
    mask = torch.isfinite(targets)
    diff = preds - torch.where(mask, targets, torch.zeros_like(targets))
    return {
        "squared_error": (diff.pow(2) * mask).sum(dim=(0, 1)),
        "abs_error": (diff.abs() * mask).sum(dim=(0, 1)),
        "count": mask.sum(dim=(0, 1)),
        "rounded_correct": ((preds.round() == targets.round()) * mask).sum(dim=(0, 1)),
        "rounded_abs": ((preds.round() - targets.round()).abs() * mask).sum(dim=(0, 1)),
    }

In [ ]:
metrics = {
    'squared_error': torch.zeros(out_timesteps, device=device),
    'abs_error': torch.zeros(out_timesteps, device=device),
    'count': torch.zeros(out_timesteps, device=device),
    'rounded_correct': torch.zeros(out_timesteps, device=device),
    'rounded_abs': torch.zeros(out_timesteps, device=device),
}

level_metrics_accum = {
    'squared_error': torch.zeros(out_timesteps, 6, device=device),
    'abs_error': torch.zeros(out_timesteps, 6, device=device),
    'count': torch.zeros(out_timesteps, 6, device=device),
    'rounded_correct': torch.zeros(out_timesteps, 6, device=device),
}

In [ ]:
start_time = time.perf_counter()
with torch.no_grad():
    for i, batch in tqdm(enumerate(traffic_dataloader), desc="Evaluando"):
        if i < max_batches_eval:
            batch = {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }
            static_inputs = batch["x_static"][0]
            dynamic_seq_inputs = (
                batch["x_dynamic_past"].permute(0, 2, 1, 3).contiguous()
            )

            # Concatenate weather features if available
            if "x_weather" in batch and batch["x_weather"] is not None:
                x_weather = batch["x_weather"].permute(0, 2, 1, 3).contiguous()
                dynamic_seq_inputs = torch.cat([dynamic_seq_inputs, x_weather], dim=-1)

            future_known_inputs = (
                batch["x_dynamic_future"].permute(0, 2, 1, 3).contiguous()
            )
            edge_index = batch["edge_index"]
            targets = batch["y"]  # .permute(0, 2, 1)

            preds = model(
                edge_index, static_inputs, dynamic_seq_inputs, future_known_inputs
            )
            batch_metrics = evaluate_batch(preds, targets)

            for key in metrics:
                metrics[key] += batch_metrics[key].to(device)

            rounded_targets = targets.round().long()
            mask = torch.isfinite(targets)
            for horizon_idx in range(out_timesteps):
                mask_h = mask[:, :, horizon_idx]
                levels = rounded_targets[:, :, horizon_idx][mask_h]
                preds_h = preds[:, :, horizon_idx][mask_h]
                targets_h = targets[:, :, horizon_idx][mask_h]
                err_sq = (preds_h - targets_h) ** 2
                err_abs = (preds_h - targets_h).abs()
                exact = (preds_h.round() == targets_h.round()).float()

                for level in range(6):
                    level_mask = levels == level
                    count = level_mask.sum()
                    if count > 0:
                        level_metrics_accum["count"][horizon_idx, level] += count
                        level_metrics_accum["squared_error"][horizon_idx, level] += (
                            err_sq[level_mask].sum()
                        )
                        level_metrics_accum["abs_error"][horizon_idx, level] += err_abs[
                            level_mask
                        ].sum()
                        level_metrics_accum["rounded_correct"][horizon_idx, level] += (
                            exact[level_mask].sum()
                        )
        else:
            break

elapsed = time.perf_counter() - start_time
print(f"Evaluación completada en {elapsed:.1f} segundos")

In [ ]:
counts = metrics["count"].cpu()
valid = counts > 0

per_step = {
    "horizon": list(range(out_timesteps)),
    "mse": torch.where(valid, metrics["squared_error"].cpu() / counts.clamp_min(1), torch.nan).tolist(),
    "mae": torch.where(valid, metrics["abs_error"].cpu() / counts.clamp_min(1), torch.nan).tolist(),
    "accuracy": torch.where(valid, metrics["rounded_correct"].cpu() / counts.clamp_min(1), torch.nan).tolist(),
    "rounded_abs": torch.where(valid, metrics["rounded_abs"].cpu() / counts.clamp_min(1), torch.nan).tolist(),
}

summary_df = pd.DataFrame(per_step)
summary_df


In [ ]:
per_step_level = []
for horizon_idx in range(out_timesteps):
    for level in range(6):
        count = level_metrics_accum['count'][horizon_idx, level]
        if count > 0:
            mse = (level_metrics_accum['squared_error'][horizon_idx, level] / count).item()
            mae = (level_metrics_accum['abs_error'][horizon_idx, level] / count).item()
            acc = (level_metrics_accum['rounded_correct'][horizon_idx, level] / count).item()
        else:
            mse = float('nan')
            mae = float('nan')
            acc = float('nan')
        per_step_level.append({
            'horizon': horizon_idx,
            'level': level,
            'mse': mse,
            'mae': mae,
            'accuracy': acc,
            'count': count.item(),
        })
per_step_level_df = pd.DataFrame(per_step_level)
per_step_level_df.pivot(index = 'horizon', columns = 'level', values = 'mse')

In [ ]:
overall_level_metrics = []
for level in range(6):
    count = level_metrics_accum['count'][:, level].sum()
    if count > 0:
        mse = (level_metrics_accum['squared_error'][:, level].sum() / count).item()
        mae = (level_metrics_accum['abs_error'][:, level].sum() / count).item()
        acc = (level_metrics_accum['rounded_correct'][:, level].sum() / count).item()
    else:
        mse = float('nan')
        mae = float('nan')
        acc = float('nan')
    overall_level_metrics.append({'level': level, 'mse': mse, 'mae': mae, 'accuracy': acc})
level_df = pd.DataFrame(overall_level_metrics)
level_df

In [ ]:
overall_results = {
    'overall_mse': (metrics['squared_error'].sum() / metrics['count'].sum().clamp_min(1)).item(),
    'overall_mae': (metrics['abs_error'].sum() / metrics['count'].sum().clamp_min(1)).item(),
    'overall_accuracy': (metrics['rounded_correct'].sum() / metrics['count'].sum().clamp_min(1)).item(),
    'overall_rounded_abs': (metrics['rounded_abs'].sum() / metrics['count'].sum().clamp_min(1)).item(),
}
overall_results

## Reconstruct Predictions to DataFrame


In [ ]:
# Metadata path - this contains the time_idx_map for timestamp reconstruction
metadata_path = f"{sparse_data_dir}/jams_dataset_metadata.pkl"

# Get the batch indices for the last evaluated batch
# (In a real scenario, this should be tracked during the evaluation loop)
last_batch_idx = min(max_batches_eval, len(traffic_dataloader)) - 1
batch_indices = list(range(last_batch_idx * batch_size, (last_batch_idx + 1) * batch_size))

# Reconstruct predictions to DataFrame
df_predictions = reconstruct_predictions(
    predictions=preds,
    dataset=traffic_dataset,
    batch_indices=batch_indices,
    metadata_path=metadata_path,
    value_column_name="predicted_level",
)

print(f"Reconstructed predictions shape: {df_predictions.shape}")
df_predictions.sample(len(df_predictions)).head(20)